# TP2 - Etapa 3: Pipeline de Deteccion y Clasificacion

Notebook de la Etapa 3: deteccion de perros con YOLOv8n, recorte de regiones,
clasificacion de cada perro detectado con el modelo entrenado en la Etapa 2,
y visualizacion de resultados sobre la imagen original.

## Equipo
- Gianfranco Frattini
- Matias Prado

## 1. Configuracion del entorno

Se clona el fork, instalan dependencias y configura el entorno. Se requiere la Etapa 2
completada (checkpoints en `models/`).

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/GianfrancoFrattini/tuia-dog-recognition-app.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone $REPO_URL proyecto
    %cd proyecto
    !pip install -q pydantic-settings python-dotenv kagglehub ultralytics
    ROOT = Path.cwd()
else:
    ROOT = Path.cwd()

os.environ["USE_PGVECTOR"] = "false"
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"YOLO model: {settings.yolo_model}")
print(f"YOLO conf threshold: {settings.yolo_conf_threshold}")
print(f"YOLO dog class id: {settings.yolo_dog_class_id}")
print(f"checkpoints: {settings.model_path}")
print(f"  resnet18: {(settings.model_path / settings.resnet18_model_name).exists()}")
print(f"  cnn_custom: {(settings.model_path / settings.cnn_custom_model_name).exists()}")

## 2. Descargar el dataset (si falta)

Se necesitan imagenes de test para probar el pipeline.

In [ ]:
if not (settings.dataset_path / 'test').is_dir():
    !python scripts/download_dataset.py
!ls data/dataset/test/ | head -5

## 3. Cargar el modelo de deteccion (YOLOv8n)

Se usa YOLOv8n pre-entrenado en COCO (80 clases). La clase `dog` es la ID 16.
No es necesario entrenar el detector — se usa directo con los pesos pre-entrenados.

In [ ]:
from lib.services.detection_service import DetectionService
from lib.services.classifier_service import ClassifierService
from lib.bootstrap import build_classifier, build_detection

classifier = build_classifier(settings)
classifier.set_active_model('resnet18_finetuned')
detection = build_detection(settings, classifier)

# Verificar que YOLO se carga correctamente
from ultralytics import YOLO
yolo = YOLO(settings.yolo_model)
print(f'YOLO cargado: {settings.yolo_model}')
print(f'Clases COCO: {len(yolo.names)} (dog = {settings.yolo_dog_class_id})')

## 4. Deteccion de perros en imagenes de test

Se ejecuta `detect_dogs` sobre imagenes del split test. La funcion devuelve una lista
de `((x1, y1, x2, y2), confidence)` en pixeles, filtrando solo la clase `dog` (16)
y por el threshold de confianza configurado.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

test_root = settings.dataset_path / 'test'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

# Tomar algunas imagenes de razas distintas para probar
sample_images = []
for breed_dir in sorted(test_root.iterdir())[:6]:
    if breed_dir.is_dir():
        for img_path in sorted(breed_dir.iterdir())[:2]:
            if img_path.suffix.lower() in IMAGE_SUFFIXES:
                sample_images.append((img_path, breed_dir.name))

print(f'Se probaran {len(sample_images)} imagenes de test')
for img_path, breed in sample_images[:6]:
    image = cv2.imread(str(img_path.resolve()))
    detections = detection.detect_dogs(image)
    print(f'  {breed}/{img_path.name}: {len(detections)} perro(s) detectado(s)')

## 5. Visualizacion de bounding boxes

Se dibujan los bounding boxes detectados por YOLO sobre cada imagen, junto con el
score de confianza de la deteccion.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (img_path, breed) in enumerate(sample_images[:6]):
    image = cv2.imread(str(img_path.resolve()))
    detections = detection.detect_dogs(image)
    
    vis = image.copy()
    for (x1, y1, x2, y2), det_score in detections:
        cv2.rectangle(vis, (x1, y1), (x2, y2), (80, 220, 80), 3)
        label = f'dog {det_score:.2f}'
        cv2.putText(vis, label, (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (80, 220, 80), 2)
    
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'{breed} ({len(detections)} det)', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Deteccion de perros con YOLOv8n — bounding boxes', fontsize=14)
plt.tight_layout()
plt.savefig('training_results/detection_examples.png', dpi=150)
plt.show()

## 6. Pipeline completo: deteccion + clasificacion

Para cada perro detectado se recorta la region (crop), se clasifica con el modelo
entrenado de la Etapa 2 (`classify_detected_dog`) y se anota la raza predicha + score
sobre la imagen.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (img_path, true_breed) in enumerate(sample_images[:6]):
    image = cv2.imread(str(img_path.resolve()))
    detections = detection.detect_dogs(image)
    
    vis = image.copy()
    for (x1, y1, x2, y2), det_score in detections:
        crop = image[y1:y2, x1:x2]
        breed_pred, breed_score = detection.classify_detected_dog(crop)
        
        cv2.rectangle(vis, (x1, y1), (x2, y2), (80, 220, 80), 3)
        label = f'{breed_pred} cls:{breed_score:.2f} det:{det_score:.2f}'
        cv2.putText(vis, label, (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (80, 220, 80), 2)
    
    color = 'green' if detections else 'red'
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    correct = ''
    if detections:
        crop = image[detections[0][0][1]:detections[0][0][3], detections[0][0][0]:detections[0][0][2]]
        pred, _ = detection.classify_detected_dog(crop)
        correct = ' ✓' if pred == true_breed else ' ✗'
    axes[i].set_title(f'Real: {true_breed}{correct}', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Pipeline completo: deteccion + clasificacion de raza', fontsize=14)
plt.tight_layout()
plt.savefig('training_results/pipeline_examples.png', dpi=150)
plt.show()

## 7. Evaluacion del pipeline sobre test

Se mide la accuracy del pipeline completo (deteccion + clasificacion) sobre el split
test: para cada imagen se detecta el perro, se clasifica el recorte y se compara la
raza predicha con la raza real.

In [ ]:
correct = 0
total = 0
no_detection = 0
errors = []

for breed_dir in sorted(test_root.iterdir()):
    if not breed_dir.is_dir():
        continue
    for img_path in sorted(breed_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        image = cv2.imread(str(img_path.resolve()))
        if image is None:
            continue
        detections = detection.detect_dogs(image)
        if not detections:
            no_detection += 1
            continue
        # Usar la deteccion con mayor confianza
        best = max(detections, key=lambda d: d[1])
        x1, y1, x2, y2 = best[0]
        crop = image[y1:y2, x1:x2]
        pred, score = detection.classify_detected_dog(crop)
        total += 1
        if pred == breed_dir.name:
            correct += 1
        else:
            errors.append({'true': breed_dir.name, 'pred': pred, 'score': score})

accuracy = correct / total if total else 0.0
detection_rate = total / (total + no_detection)
print(f'Pipeline completo (YOLO + ResNet18 fine-tuned):')
print(f'  Imagenes con deteccion: {total}/{total + no_detection} ({detection_rate:.1%})')
print(f'  Accuracy de clasificacion: {correct}/{total} = {accuracy:.4f}')
print(f'  Sin deteccion (YOLO no detecto perro): {no_detection}')

In [ ]:
# Analisis de errores del pipeline
import pandas as pd
if errors:
    err_df = pd.DataFrame(errors)
    confusion = err_df.groupby(['true', 'pred']).size().reset_index(name='count')
    confusion = confusion.sort_values('count', ascending=False).head(15)
    print('Top 15 confusiones del pipeline (real -> predicha):')
    print(confusion.to_string(index=False))

## 8. Deteccion en escenas complejas (multiples perros)

Se prueba el pipeline con imagenes que contienen multiples perros u otros objetos.
Para esto se pueden usar imagenes descargadas de internet en `data/external/`.

In [ ]:
# Si hay imagenes externas en data/external/, probarlas aca.
# Sino, se puede subir una imagen manualmente en Colab:
# from google.colab import files; uploaded = files.upload()

external = settings.dataset_path.parent / 'external'
if external.is_dir():
    ext_images = list(external.rglob('*.jpg'))[:6]
    if ext_images:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        for i, img_path in enumerate(ext_images[:6]):
            image = cv2.imread(str(img_path))
            detections = detection.detect_dogs(image)
            vis = image.copy()
            for (x1, y1, x2, y2), det_score in detections:
                crop = image[y1:y2, x1:x2]
                breed_pred, breed_score = detection.classify_detected_dog(crop)
                cv2.rectangle(vis, (x1, y1), (x2, y2), (80, 220, 80), 3)
                label = f'{breed_pred} cls:{breed_score:.2f} det:{det_score:.2f}'
                cv2.putText(vis, label, (x1, max(0, y1-10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (80, 220, 80), 2)
            axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            axes[i].set_title(f'{img_path.name} ({len(detections)} perros)')
            axes[i].axis('off')
        plt.tight_layout()
        plt.savefig('training_results/external_detection.png', dpi=150)
        plt.show()
    else:
        print('No se encontraron imagenes en data/external/')
else:
    print(f'Conjunto externo: ubicarlo en {external} (imagenes de internet con multiples perros).')

## 9. Conclusiones de la Etapa 3

1. **YOLOv8n** detecta perros de forma robusta en imagenes del dataset (tasa de
   deteccion alta) sin necesidad de entrenamiento, gracias a los pesos COCO.
2. **El pipeline completo** (deteccion + clasificacion) logra una accuracy cercana
   a la del clasificador solo, lo que indica que los recortes de YOLO son lo
   suficientemente buenos para que el clasificador funcione correctamente.
3. **Los errores** del pipeline provienen de dos fuentes:
   - Falsos negativos de deteccion (YOLO no detecta el perro, especialmente en
     imagenes donde el perro ocupa poco espacio o esta parcialmente ocluido).
   - Confusiones de clasificacion entre razas morfologicamente similares (los
     mismos errores que se observan en la Etapa 2).
4. **En escenas con multiples perros**, el pipeline detecta y clasifica cada uno
   independientemente, generando un bounding box + raza + score por perro.
5. **La integracion** con el frontend Gradio permite subir imagenes y ver los
   resultados visuales (bounding boxes + razas) directamente en el navegador.